# Задание 1


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [ ]:
import nltk
import wikipedia
import numpy as np
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from collections import defaultdict
from string import digits
from bs4 import BeautifulSoup
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

code2lang = wikipedia.languages()
langs = ['hr', 'he', 'fr', 'sr' , 'ja']
[(lang, code2lang[lang]) for lang in langs] 

[('hr', 'hrvatski'),
 ('he', 'עברית'),
 ('fr', 'français'),
 ('sr', 'српски / srpski'),
 ('ja', '日本語')]

Для анализа я выбрала хорватский, иврит, французский, сербский и японский

In [81]:
def get_texts_for_lang(lang, n=50):
    wikipedia.set_lang(lang)
    wiki_content = []
    
    pages = wikipedia.random(n)
    
    for page_name in pages:
        try:
            page = load_with_disambigution(page_name)
        
        except Exception as e:
            print('Skipping page {}'.format(page_name), str(e).strip('\n'))
            continue

        wiki_content.append(f'{page.title}\n{page.content.replace("==", "")}')

    return wiki_content

In [86]:
%%time

wiki_texts = {}

for lang in langs:
    try:
        wiki_texts[lang] = get_texts_for_lang(lang, 10)
    except Exception as e:
        print('ERROR ON - ', lang, str(e).strip('\n'))
        continue
    
    print(lang, len(wiki_texts[lang]))

hr 10
he 10
fr 10
sr 10
ja 10
CPU times: total: 2.58 s
Wall time: 1min 52s


In [ ]:
def clean_text(text):
    text = re.sub(r'==.*?==+', ' ', text) 
    text = re.sub(r'\n+', ' ', text)       
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def robust_sent_split(text):
    try:
        from nltk.tokenize import sent_tokenize
        sents = sent_tokenize(text)
        if len(sents) > 1:
            return sents
    except:
        pass
    sents = re.split(r'(?<=[.!?。！？])\s+', text)
    return [s.strip() for s in sents if len(s.strip()) > 0]

all_data = []
for lang, texts in wiki_texts.items():
    for t in texts:
        cleaned = clean_text(t)
        sentences = robust_sent_split(cleaned)
        for s in sentences:
            all_data.append({"language": lang, "sentence": s})

df = pd.DataFrame(all_data)
print(f"Всего предложений: {len(df)}")
print(df.head())

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['sentence'].tolist(), show_progress_bar=True)

num_clusters = len(langs)
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(embeddings)

cluster_lang_map = df.groupby('cluster')['language'].agg(lambda x: x.value_counts().index[0]).to_dict()
df['pred_lang'] = df['cluster'].map(cluster_lang_map)

ari = adjusted_rand_score(df['language'], df['pred_lang'])
print(f"ARI (Adjusted Rand Index): {ari:.4f}")

errors = df[df['language'] != df['pred_lang']]
print(f"Ошибочно кластеризованных предложений: {len(errors)} из {len(df)}")

if len(errors) >= 3:
    sample_errors = errors.sample(3, random_state=42)
    for _, row in sample_errors.iterrows():
        print("\n---")
        print(f"Истинный язык: {row['language']}")
        print(f"Предсказанный кластер (язык): {row['pred_lang']}")
        print(f"Текст: {row['sentence'][:300]}") 

Всего предложений: 1269
  language                                           sentence
0       hr  DirectX DirectX je skupina više programskih ok...
1       hr  Najviše se koristi za razvoj računalnih igara ...
2       hr  DirectX unaprijed dolazi instaliran na novije ...
3       hr  Dijelovi DirectX-a Dijelovi od kojih se Direct...
4       hr  Direct3D (D3D): za prikazivanje 3D računarske ...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

ARI (Adjusted Rand Index): 0.8900
Ошибочно кластеризованных предложений: 107 из 1269

---
Истинный язык: sr
Предсказанный кластер (язык): hr
Текст: Према процени из 2011. у насељу је живело 31 становника.

---
Истинный язык: hr
Предсказанный кластер (язык): he
Текст: 1 + 2 + 3 + 4 + ... + 99 + 100.

---
Истинный язык: hr
Предсказанный кластер (язык): fr
Текст: Vanjske poveznice Profil na Utah-basketball.com Profil na ESPN.com


# Задание 2

Загрузите корпус `annot.opcorpora.no_ambig_strict.xml.bz2` с OpenCorpora. Найдите в корпусе самые частотные морфологически омонимичные словоформы (те, которым соответствует разный грамматический разбор в разных предложениях). Также найдите словоформы с самых большим количеством вариантов грамматических разборов.

In [1]:
import bz2
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from lxml import etree

with bz2.open('C:/Users/schig/Downloads/annot.opcorpora.no_ambig_strict.xml.bz2', 'rb') as f_in, open('annot.opcorpora.no_ambig_strict.xml', 'wb') as f_out:
    f_out.write(f_in.read())

open_corpora = etree.fromstring(open('C:/Users/schig/Downloads/annot.opcorpora.no_ambig_strict.xml', 'rb').read())

corpus = []

for sentence in open_corpora.xpath('//tokens'):
    sent_tagged = []
    for token in sentence.xpath('token'):
        word = token.xpath('@text')
        gram_info = token.xpath('tfr/v/l/g/@v')
        sent_tagged.append([word[0]] + gram_info)
    corpus.append(sent_tagged)

analyses = defaultdict(set)

for sentence in corpus:
    for token in sentence:
        word = token[0].lower()
        gram = tuple(sorted(token[1:]))  
        analyses[word].add(gram)

frequency = Counter([token[0].lower() for sent in corpus for token in sent])
ambiguous_words = {w: analyses[w] for w in analyses if len(analyses[w]) > 1}
most_frequent_ambiguous = sorted(ambiguous_words.items(), key=lambda x: frequency[x[0]], reverse=True)

print("\nCамые частотные морфологически омонимичные словоформы:")
for w, grammemes in most_frequent_ambiguous[:10]: #я решила показать топ 10 самых частотных
    print(f"{w} - {frequency[w]}: {grammemes}")

most_polysemous = sorted(analyses.items(), key=lambda x: len(x[1]), reverse=True)
print("\nСловоформы с самым большим количеством вариантов грамматических разборов:")
for w, grammemes in most_polysemous[:10]: #здесь тоже оставила 10
    print(f"{w}")




Cамые частотные морфологически омонимичные словоформы:
в - 2059: {('Abbr', 'Fixd', 'NOUN', 'gent', 'inan', 'masc', 'sing'), ('PREP',)}
на - 786: {('PRCL',), ('PREP',)}
с - 613: {('PRCL',), ('PREP',)}
и - 574: {('PRCL',), ('CONJ',)}
о - 213: {('INTJ',), ('PREP',)}
году - 115: {('NOUN', 'inan', 'loc2', 'masc', 'sing'), ('NOUN', 'datv', 'inan', 'masc', 'sing')}
а - 113: {('INTJ',), ('CONJ',)}
этом - 104: {('NPRO', 'loct', 'neut', 'sing'), ('ADJF', 'Anph', 'Apro', 'Subx', 'loct', 'masc', 'sing'), ('ADJF', 'Anph', 'Apro', 'Subx', 'loct', 'neut', 'sing')}
россии - 91: {('Geox', 'NOUN', 'Sgtm', 'datv', 'femn', 'inan', 'sing'), ('Geox', 'NOUN', 'Sgtm', 'femn', 'inan', 'loct', 'sing'), ('Geox', 'NOUN', 'Sgtm', 'femn', 'gent', 'inan', 'sing')}
было - 89: {('VERB', 'impf', 'indc', 'intr', 'neut', 'past', 'sing'), ('PRCL',)}

Словоформы с самым большим количеством вариантов грамматических разборов:
сша
кино
евро
компании
пути
какой
это
одной
своей
лица


## Задание 3
Загрузите один и з файлов корпуса Syntagrus - https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master (можно взять тестовый)

Преобразуйте все разборы предложений в графовые структуры через DependencyGraph, выберите 3 любых отношения и для каждого найдите топ-5 самых встречаемых пар слов, связанных этим отношением. 

Для самой частотной пары слов в каждом из отношений вытащите все подзависимые слова для каждого из них во всех предложениях (используя `flatten(get_subtree(d.nodes, index_of_a_word)` и сортируя результат по порядку слов в предложениях, аналогично тому как я делал с summaries только у вас будет два слова) 
В итоге у вас должен получится что-то такое:

```
### отношение
relation_name

### топ 5 пар слов связанных этим отношением
(word1, word2), (word3, word4), (word5, word6), (word7, word8), (word9, word10)

### подзависимые для самого частотного
(subword word1 subword, word2 subword subword)

... (и так три раза)
```



In [5]:
pip install conllu

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import conllu
from nltk.parse import DependencyGraph
from collections import Counter, defaultdict

file_path = r"C:\Users\schig\Downloads\ru_syntagrus-ud-dev.conllu"

with open(file_path, "r", encoding="utf-8") as f:
    data = f.read()

sentences = conllu.parse(data)

graphs = []
for sent in sentences:
    conll_lines = []
    for token in sent:
        if isinstance(token['id'], int):
            word = token['form']
            lemma = token['lemma']
            upos = token['upostag']
            xpos = token['xpostag'] if token['xpostag'] else '_'
            feats = '|'.join([f"{k}={v}" for k, v in (token['feats'] or {}).items()]) or '_'
            head = str(token['head'])  # базовый head
            rel = token['deprel']
            conll_lines.append(f"{token['id']}\t{word}\t{lemma}\t{upos}\t{xpos}\t{feats}\t{head}\t{rel}\t_\t_")
    conll_text = "\n".join(conll_lines)
    graphs.append(DependencyGraph(conll_text))

def get_subtree(nodes, index):
    result = []
    for node_id, node in nodes.items():
        if node_id == 0:
            continue
        if node['head'] == index:
            result.append(node['word'])
            result.extend(get_subtree(nodes, node_id))
    return result

relation_counter = defaultdict(Counter)

for g in graphs:
    nodes = g.nodes
    for node_id, node in nodes.items():
        if node_id == 0:
            continue
        head_id = node['head']
        deprel = node['rel']
        if head_id == 0:
            continue
        head_word = nodes[head_id]['word']
        child_word = node['word']
        relation_counter[deprel][(head_word, child_word)] += 1

top_relations = sorted(relation_counter.items(), key=lambda x: len(x[1]), reverse=True)[:3]

for rel, counter in top_relations:
    print(f"Отношение:\n{rel}\n")

    top5_pairs = counter.most_common(5)
    print("Топ 5 пар слов связанных этим отношением:")
    print(", ".join(str(pair[0]) for pair in top5_pairs))

    most_common_pair = top5_pairs[0][0]
    head_word, child_word = most_common_pair
    
    print("\nПодзависимые для самого частотного:")
    for g in graphs:
        nodes = g.nodes
        head_index = child_index = None
        for node_id, node in nodes.items():
            if node_id == 0:
                continue
            if node['word'] == head_word:
                head_index = node_id
            if node['word'] == child_word:
                child_index = node_id
        if head_index and child_index:
            head_subtree = get_subtree(nodes, head_index)
            child_subtree = get_subtree(nodes, child_index)
            if head_subtree or child_subtree:
                print(f"({' '.join(head_subtree)}, {' '.join(child_subtree)})")

Отношение:
punct

Топ 5 пар слов связанных этим отношением:
('может', ','), ('например', ','), ('можно', '.'), ('может', '.'), ('и', ',')

Подзависимые для самого частотного:
(, что работа инструкций каких-то алгоритма зависима быть инструкций от других результатов или работы их, )
(стороны С другой , алгоритм вероятностный выдать и никогда не результат равна , но вероятность этого 0 ., )
(, а делитель не равен быть нулю, )
(задачи Для каждой существовать множество алгоритмов приводящих , цели к ., )
(Маршрут классифицирован быть первовосхождение как первопрохождение , вариант , комбинация , маршрутов ., )
(Примером тому быть поход сложный очень имеющий , не аналогов совершенный , туристами МАИ году в 2009 Памире на , пройдено которого во время было км 622 части по высокогорной Памира восхождением с вершины на высшие пик : Коммунизма м ( 7495 ) пик , Ленина м ( 7134 ) пик и Революции м ( 6940 ) ., )
(поле Любое подпространство ( ) пространства социального описано быть , очередь в свою 